In [ ]:
# Google Maps Agent — Vertex AI + ADK

Google Maps Agent using **Google ADK** with **Vertex AI (Gemini)**.  
Directly calls Google Maps APIs for directions, place search, and place details.

Configure values in `.env`:

| Variable | Purpose |
|---|---|
| `GOOGLE_MAPS_API_KEY` | Google Maps API Key |
| `GOOGLE_CLOUD_PROJECT` | GCP Project ID |
| `GOOGLE_CLOUD_LOCATION` | GCP Region |
| `MODEL_ID` | Gemini model ID |
| `GOOGLE_GENAI_USE_VERTEXAI` | Set to `TRUE` |

In [ ]:
!pip install google-adk requests google-genai python-dotenv --quiet

In [ ]:
import os
import json
import requests
from typing import Dict, Any, Optional
from dotenv import load_dotenv
from google.adk.agents import LlmAgent
from google.adk.tools import FunctionTool
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

load_dotenv()

GOOGLE_MAPS_API_KEY = os.environ["GOOGLE_MAPS_API_KEY"]
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = os.environ["GOOGLE_GENAI_USE_VERTEXAI"]
os.environ["GOOGLE_CLOUD_PROJECT"] = os.environ["GOOGLE_CLOUD_PROJECT"]
os.environ["GOOGLE_CLOUD_LOCATION"] = os.environ["GOOGLE_CLOUD_LOCATION"]
MODEL_ID = os.environ["MODEL_ID"]

print(f"Project : {os.environ['GOOGLE_CLOUD_PROJECT']}")
print(f"Region  : {os.environ['GOOGLE_CLOUD_LOCATION']}")
print(f"Model   : {MODEL_ID}")
print(f"Maps Key: {'***' + GOOGLE_MAPS_API_KEY[-4:] if len(GOOGLE_MAPS_API_KEY) > 8 else 'Not set'}")

In [ ]:
def get_directions(origin: str, destination: str, mode: str = "driving") -> dict:
    """Get step-by-step directions from origin to destination with distance and duration."""
    base_url = "https://maps.googleapis.com/maps/api/directions/json"
    try:
        params = {
            "origin": origin,
            "destination": destination,
            "mode": mode,
            "key": GOOGLE_MAPS_API_KEY,
        }

        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        if data["status"] != "OK":
            return {"error": data.get("error_message", data["status"])}

        route = data["routes"][0]
        leg = route["legs"][0]

        steps = []
        for i, step in enumerate(leg["steps"], 1):
            instruction = (
                step["html_instructions"]
                .replace("<b>", "")
                .replace("</b>", "")
                .replace('<div style="font-size:0.9em">', " - ")
                .replace("</div>", "")
            )
            steps.append({
                "step": i,
                "instruction": instruction,
                "distance": step["distance"]["text"],
                "duration": step["duration"]["text"],
            })

        return {
            "origin": leg["start_address"],
            "destination": leg["end_address"],
            "distance": leg["distance"]["text"],
            "duration": leg["duration"]["text"],
            "mode": mode,
            "steps": steps,
        }

    except requests.exceptions.RequestException as e:
        return {"error": f"API Error: {str(e)}"}
    except (KeyError, IndexError) as e:
        return {"error": f"Data parsing error: {str(e)}"}

In [ ]:
def search_places(query: str, location: Optional[str] = None) -> dict:
    """Search for places, restaurants, hotels, attractions, or any point of interest."""
    base_url = "https://maps.googleapis.com/maps/api/place/textsearch/json"
    try:
        search_query = f"{query} in {location}" if location else query

        params = {
            "query": search_query,
            "key": GOOGLE_MAPS_API_KEY,
        }

        response = requests.get(base_url, params=params, timeout=15)
        response.raise_for_status()
        data = response.json()

        if data["status"] != "OK":
            return {"error": data.get("error_message", data["status"])}

        results = data["results"][:5]

        places = []
        for place in results:
            places.append({
                "name": place["name"],
                "address": place.get("formatted_address", "Address not available"),
                "rating": place.get("rating", "N/A"),
                "types": place.get("types", [])[:3],
            })

        return {"query": search_query, "results": places}

    except Exception as e:
        return {"error": str(e)}

In [ ]:
def get_place_details(place_name: str) -> dict:
    """Get detailed information about a place including hours, phone, website, and reviews."""
    search_url = "https://maps.googleapis.com/maps/api/place/textsearch/json"
    details_url = "https://maps.googleapis.com/maps/api/place/details/json"

    try:
        search_params = {"query": place_name, "key": GOOGLE_MAPS_API_KEY}
        search_response = requests.get(search_url, params=search_params, timeout=10)
        search_response.raise_for_status()
        search_data = search_response.json()

        if search_data["status"] != "OK" or not search_data["results"]:
            return {"error": f"Could not find place: {place_name}"}

        place_id = search_data["results"][0]["place_id"]

        details_params = {
            "place_id": place_id,
            "fields": "name,formatted_address,formatted_phone_number,website,rating,opening_hours,reviews,price_level,url",
            "key": GOOGLE_MAPS_API_KEY,
        }

        details_response = requests.get(details_url, params=details_params, timeout=10)
        details_response.raise_for_status()
        details_data = details_response.json()

        if details_data["status"] != "OK":
            return {"error": details_data.get("error_message", details_data["status"])}

        place = details_data["result"]

        hours = []
        if "opening_hours" in place:
            hours = place["opening_hours"].get("weekday_text", [])

        reviews = []
        for review in place.get("reviews", [])[:3]:
            reviews.append({
                "author": review.get("author_name", "Anonymous"),
                "rating": review.get("rating", "N/A"),
                "text": review.get("text", "")[:200],
                "time": review.get("relative_time_description", ""),
            })

        return {
            "name": place.get("name", place_name),
            "address": place.get("formatted_address", "N/A"),
            "phone": place.get("formatted_phone_number", "N/A"),
            "website": place.get("website", "N/A"),
            "rating": place.get("rating", "N/A"),
            "price_level": place.get("price_level", "N/A"),
            "google_maps_url": place.get("url", "N/A"),
            "hours": hours,
            "reviews": reviews,
        }

    except requests.exceptions.RequestException as e:
        return {"error": f"API Error: {str(e)}"}
    except (KeyError, IndexError) as e:
        return {"error": f"Data parsing error: {str(e)}"}

In [ ]:
google_maps_agent = LlmAgent(
    name="google_maps_agent",
    model=MODEL_ID,
    description="Google Maps assistant for directions, place search, and place details.",
    instruction="""You are a helpful Google Maps assistant. You can:
1. Get Directions — step-by-step navigation between two locations.
2. Search Places — find restaurants, hotels, attractions, or any POI.
3. Get Place Details — retrieve hours, phone, website, reviews for a place.

Use the appropriate tool for each request. Present results clearly.
""",
    tools=[get_directions, search_places, get_place_details],
)

print(f"Agent '{google_maps_agent.name}' created with {len(google_maps_agent.tools)} tools")

In [ ]:
import asyncio

APP_NAME = "google_maps_app"
USER_ID = "user_1"
SESSION_ID = "session_001"

session_service = InMemorySessionService()

runner = Runner(
    agent=google_maps_agent,
    app_name=APP_NAME,
    session_service=session_service,
)


async def call_agent(user_message: str):
    """Send a message to the agent and print the response."""
    print(f"\n{'='*60}")
    print(f"USER: {user_message}")
    print(f"{'='*60}")

    content = types.Content(
        role="user",
        parts=[types.Part.from_text(user_message)],
    )

    final_response = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=content,
    ):
        if event.is_final_response():
            for part in event.content.parts:
                if part.text:
                    final_response += part.text

    print(f"\nAGENT:\n{final_response}")
    return final_response

In [ ]:
await call_agent("How do I get from Times Square, New York to Central Park by walking?")

In [ ]:
await call_agent("Find the best Italian restaurants in San Francisco")

In [ ]:
await call_agent("Tell me everything about the Eiffel Tower — hours, reviews, website")

In [ ]:
while True:
    user_input = input("Ask the Maps Agent (or 'quit'): ").strip()
    if not user_input or user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    await call_agent(user_input)